<a href="https://colab.research.google.com/github/zhangling297/deep-learning-with-python-notebooks/blob/master/Copy_of_Fully_connected_NN_For_Binary_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

fully- connected NN for binary classification - using heart disease data - upload the data, subset the columns to the 3 objects that we are interested in, and then standardize the data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

#Load the heart disease CSV file into a DataFrame
df_heart_disease = pd.read_csv('/heart_disease.csv')

print(df_heart_disease)

In [ ]:

# Define the predictor variabies and target variable form the subset
X = df_heart_disease[['chol', 'age', 'trestbps']]
y = df_heart_disease['target']

#Initialize the StandardScaler
scaler = StandardScaler()

#Fit the scaler to the data and transform it

X_scaled = scaler.fit_transform(X)

# Convert the scaled array back to a DataFrame  for easier viewing
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

#check what it looks like
display(X_scaled.head())

**Now we want to define the model. The following code block is blank for you to define a model in Torch: Attempt to write a class for a fully-connected NN with one hidden layer with 8 neurons**.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class FullyConnectedNN(nn.Module):
  def_init_(self, input_size, hidden_size=8):
    super(FullyConnectedNN, self)._init_()
    self.fc1 = nn.Linear(input_size, hidden_size)
    self.fc2 = nn.Linear(hidden_size, 1)
    self.sigmoid = nn.Sigmoid()
    self.relu = nn.ReLu()

  def forward(self, x):
    z1 = self.fc1(x)
    a1 = self.relu(z1)
    z2 = self.fc2(a1)
    return self.sigmoid(z2)


# Now create the training loop where you learn the parameters wiht gradient descent

In [ ]:
# To use PyTorch you must convert our data to PyTorch tensors
X_torch = torch.tensor(X_scaled.values,dtype=torch.float32)
y_torch=torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

#Initialize the model we defined above
model = FullyConnectedNN(input_size=3)

#Print the random initial values of the weights and bias
with torch.no_grad():
  print('Initial Values: Weights = {model.fc1.weight.numpy()[0].round(decimals=3)}, \
  Bias = {model.fc1.bias.numpy().round(decimals=3)}')
# Define our Loss function and algorithm for optimization (aka our 'Optimizer')
# BCELoss = Binary Cross Entropy Loss(standard for binary classification)
criterion = nn.BCELoss()

# This is the function for "Stocastic' Grandient Descent not plain Gradient Descent. Below does not add any of the Stochastic elements(randominess) so it will act exactly as what appear here
optimizer = optim.SGD(model.parameters(), lr=0.05)

# The following is the training loop, the number of"epochs"is is the number of times we want our whole dataset to propagate forward through the model. Here the number of epocs is also the number of steps of gradient descent will be made

epochs = 10_000

print()
print('Starting Training with Gradient Descent')
print()
for epoch in range(epochs):

  #pass the entire dataset X at once to the model and calcualte predictions
  y_pred = model(X_torch)

  # note here we actually calculate the cost not the loss but all PyTorch syntax calls this the loss
  loss = criterion(y_pred, y_torch)

  # These steps (1) delete any previous gradient calculations; 2) do backpropagation and 3) update the model parameters
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if (epoch +1) % 50 == 0:
    # the below line prevents Torch from remembering the following,
    # slower with larger datasets
    with torch.no_grad():
      predicted_labels = (y_pred > 0.5).float()
      accuracy = (predicted_labels == y_torch).float().mean()
      print(f'Epoch {epoch+1}: Loss = {loss.item(): 5f}, Accuracy = {accuracy.item():.2f}')
  print()
  print('Training Complete.')
  print()


In [ ]:
#Using CNN to do Binary Clasfication
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import torch
import torch.utils.data
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

In [ ]:
# this is a transform to normalize these images
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307,], std=[0.3081,])
])

# Download and load the train and test data
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
print(f"MNIST training data loaded: {len(train_dataset)} samples")
print(f"MNIST test data loaded: {len(test_dataset)} samples")

# Filter the data to make this a binary problem. We will look at multi-class next but let's start with binary.
def filter_labels_0_1(dataset):
    # Initialize an empty list to store the indices
    indices_0_1 = []

    # Iterate through the data
    for i, (image, label) in enumerate(dataset):
        # Check if the label is 0 or 1
        if label == 0 or label == 1:
            indices_0_1.append(i)

    # Create a new Subset using the original dataset and the collected indices
    filtered_dataset = torch.utils.data.Subset(dataset, indices_0_1)

    return filtered_dataset

train_dataset_binary = filter_labels_0_1(train_dataset)
test_dataset_binary = filter_labels_0_1(test_dataset)

print(f"Filtered MNIST training data loaded: {len(train_dataset_binary)} samples")
print(f"Filtered MNIST test data loaded: {len(test_dataset_binary)} samples")

In [ ]:
# Subset the filtered training data to 50% of its size
# Calculate the number of samples for 50%
total_size = len(train_dataset_binary)
subset_size = total_size // 2

# Generate random indices for the subset
all_indices = list(range(total_size))
random.shuffle(all_indices)
subset_indices = all_indices[:subset_size]

# Create the subset
train_dataset_binary_subset = torch.utils.data.Subset(train_dataset_binary, subset_indices)

print(f"Original filtered training size: {total_size}")
print(f"New 50% subset training size: {len(train_dataset_binary_subset)}")

# Re-assign to train_dataset_binary if you want to use it in the existing training loop
# train_dataset_binary = train_dataset_binary_subset

In [ ]:
# visualize our input data
def plot_random_train_image(dataset=train_dataset_binary):

    # get random index
    idx = random.randint(0, len(dataset) - 1)
    image, label = dataset[idx]
    print(f"Index: {idx}")

    # Unnormalize for visualization
    # The original transform was Normalize(mean=[0.1307], std=[0.3081])
    # Formula: unnormalized = (normalized * std) + mean
    mean = 0.1307
    std = 0.3081
    unnormalized_image = image * std + mean

    plt.imshow(unnormalized_image.squeeze(), cmap='gray_r')
    plt.title(f'Label: {label}')
    plt.axis('off')
    plt.show()

# Plot a random sample from the filtered training data
plot_random_train_image()

In [ ]:
# create a CNN class
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Layer 1: 1 input channel (grayscale), 32 output channels, 3x3 filter
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding='valid')

        # Layer 2: 32 input channels, 64 output channels, 5x5 filter
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, stride=1, padding='valid')

        # Fully connected layers
        self.fc1 = nn.Linear(22 * 22 * 32, 64) # 28 x 28 image becomes 22 x 22 after the conv layers
        self.fc2 = nn.Linear(64, 1)

        # Dropout to prevent overfitting
        self.dropout = nn.Dropout(0.1)

        # activation function
        self.ReLU = nn.ReLU()

        # sigmoid for binary classification
        self.sigmoid = nn.Sigmoid()


    def forward(self, x):
        # First Conv block
        x = self.ReLU(self.conv1(x))

        # Second Conv block
        x = self.ReLU(self.conv2(x))

        # Flatten the 3D tensor into a 1D vector for the FC layers
        x = torch.flatten(x, 1)

        # Fully connected layers with Dropout
        x = self.ReLU(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        #sigmoid function to make prediction binary
        return self.sigmoid(x)

# Instantiate the model
model = SimpleCNN()
print(model)

In [ ]:
#Build a CNN for Medical Image Classification
# Install MedMNIST

# Set up
!pip install medmnist -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
import medmnist
from medmnist import INFO

device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
#Load and Explore the dataset
data_flag = 'bloodmnist'
info = INFO[data_flag]
n_classes = len(info['label'])
class_names = list(info['label'].values())

print(f"Dataset: {data_flag}")
print(f"Number of classes: {n_classes}")
print(f"Classes: {class_names}")
print(f"Task: {info['task']}")

In [ ]:
basic_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

DataClass = getattr(medmnist, info['python_class'])

train_dataset_full = DataClass(split='train', transform=basic_transform, download=True)
val_dataset        = DataClass(split='val',   transform=basic_transform, download=True)
test_dataset       = DataClass(split='test',  transform=basic_transform, download=True)

# Use a subset to keep things fast on CPU
torch.manual_seed(42)
train_dataset = Subset(train_dataset_full, torch.randperm(len(train_dataset_full))[:1000])
val_dataset   = Subset(val_dataset, torch.randperm(len(val_dataset))[:500])
test_dataset  = Subset(test_dataset, torch.randperm(len(test_dataset))[:500])

print(f"\nTrain size: {len(train_dataset)} (subset of {len(train_dataset_full)})")
print(f"Val size:   {len(val_dataset)}")
print(f"Test size:  {len(test_dataset)}")

In [ ]:
Visionlization the data

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flatten()):
    img, label = train_dataset[i]

    # TODO: Convert the image tensor for display
    # 1. Undo normalization (multiply by 0.5, add 0.5)
    # 2. Permute dimensions from (C, H, W) to (H, W, C)
    # 3. Convert to numpy
    img_display = ______  # YOUR CODE HERE

    ax.imshow(img_display)
    ax.set_title(class_names[label.item()], fontsize=8)
    ax.axis('off')

plt.suptitle('BloodMNIST - Sample Blood Cell Images', fontsize=14)
plt.tight_layout()
plt.show()